In [ ]:
!pip install -q ddgs ollama
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

!pkill -f "ollama serve"
import time
time.sleep(2)

import subprocess, time

with open('/kaggle/working/ollama.log', 'w') as log_file:
    subprocess.Popen(['ollama', 'serve'], stdout=log_file, stderr=log_file)

time.sleep(5)
print("Service Ollama demarre")

!ollama pull gemma3:4b

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import csv
import random
import re
import time
import ollama
from ddgs import DDGS
from collections import Counter

**TESTER MA CONFIG**

**DEBUT**

In [ ]:
!nvidia-smi

In [ ]:
!ollama run gemma3:4b "Dis bonjour en une phrase"

In [ ]:
!ollama run gemma3:4b "Dis bonjour autrement"

In [ ]:
reponse = ollama.chat(model="gemma3:4b", messages=[
    {"role": "user", "content": "Dis bonjour en une phrase"}
])
print(reponse["message"]["content"])

In [ ]:
resultats = DDGS().text("Miley Cyrus Liam Hemsworth married", max_results=3)
for r in resultats:
    print(r["title"], "-", r["href"])

**FIN DES TESTS**

In [ ]:
taille = 150
m = "gemma3:4b"
var_aleatoire = 30

random.seed(var_aleatoire)

def lire_csv(chemin):
    with open(chemin, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

# fonction qui va retourner n articles du fichier csv lignes
def echantillonner(lignes, n):
    n = min(n, len(lignes))
    return random.sample(lignes, n)


In [ ]:
def chercher_preuves(titre, texte):
    # cherche sur web des infos sur le texte passé en param
    # retourne les 3 premiers resultats trouvés formatés en liste de points

    requete = titre if titre else texte[:100]
    try:
        resultats = DDGS().text(requete, max_results=3)
    except Exception as e:
        print(f"  Erreur de recherche : {e}")
        return "(aucune preuve trouvee)"

    if not resultats:
        return "(aucune preuve trouvee)"

    morceaux = []
    for r in resultats:
        morceaux.append(f"- {r.get('title', '')} : {r.get('body', '')[:200]}")
    return "\n".join(morceaux)


In [ ]:
def chercher_preuves_2(titre, texte, max_tentatives=3):
    # cherche sur web des infos sur le texte passé en param
    # retourne les 3 premiers resultats trouvés formatés en liste de points
    # difference avec l'autre, si on échoue a trouver des preuves, on essaye jusqu'a 3 tentatives

    requete = titre if titre else texte[:100]

    for tentative in range(1, max_tentatives + 1):
        try:
            # timeout augmente a 20s (au lieu de 5s par defaut)
            with DDGS(timeout=20) as ddgs:
                resultats = ddgs.text(query=requete, max_results=3)

            if not resultats:
                return "(aucune preuve trouvee)"

            morceaux = []
            for r in resultats:
                morceaux.append(f"- {r.get('title', '')} : {r.get('body', '')[:200]}")
            return "\n".join(morceaux)

        except Exception as e:
            print(f"  [!] Tentative {tentative}/{max_tentatives} echouee : {e}")
            if tentative < max_tentatives:
                # attend de plus en plus longtemps a chaque tentative (3s, 6s, 9s)
                time.sleep(3 * tentative)

    # si toutes les tentatives ont echoue
    return "(aucune preuve trouvee apres plusieurs tentatives)"


In [ ]:
def construire_prompt(titre, texte, source, preuves):
    texte_court = texte[:800]

    prompt = "Voici un article a verifier.\n\n"
    prompt = prompt + "Source : " + source + "\n"
    prompt = prompt + "Titre : " + titre + "\n"
    prompt = prompt + "Texte : " + texte_court + "\n\n"
    prompt = prompt + "Preuves trouvees sur le web concernant cette affirmation :\n"
    prompt = prompt + preuves + "\n\n"
    prompt = prompt + "En te basant sur ces preuves, verifie si l'affirmation de cet article est vraie ou fausse.\n"
    prompt = prompt + "Reponds uniquement par un seul mot : \"fake\" ou \"real\"."

    return prompt


def extraire_verdict(reponse_llm):
    texte = reponse_llm.lower()
    if re.search(r"\bfake\b", texte):
        return "fake"
    if re.search(r"\breal\b", texte):
        return "real"
    return "indetermine"

In [ ]:
def demander_verdict(titre, texte, source, preuves):

    prompt = construire_prompt(titre, texte, source, preuves)

    reponse = ollama.chat(model=m, messages=[{"role": "user", "content": prompt}])

    contenu = reponse["message"]["content"]

    return extraire_verdict(contenu)


**TEST**

In [ ]:
# On charge le fichier B et on prend juste le premier article pour tester
lignes_test = lire_csv("/kaggle/input/datasets/imenec/data-in/categorie_B_egalise.csv")
article_test = lignes_test[0]

titre = article_test.get("title", "")
texte = article_test.get("text", "")
source = article_test.get("domaine", "source inconnue")
vrai_label = article_test.get("label", "")

print("Titre :", titre)
print("Source :", source)
print("Vrai label :", vrai_label)

preuves = chercher_preuves_2(titre, texte)
print("\nPreuves trouvees :\n", preuves)

verdict = demander_verdict(titre, texte, source, preuves)
print("\nVerdict du LLM :", verdict)

**FIN de TEST**

In [ ]:
# a revoir
def test_groupe(chemin_csv, nom_groupe, chemin_sortie, taille_groupe):
    print(f"\n Traitement du groupe {nom_groupe} :")

    lignes = lire_csv(chemin_csv)
    #echantillonner est un moyen pour tester sur un groupe restreint d'Article au lieu du groupe au complet
    echantillon = echantillonner(lignes, taille_groupe)

    print(f"{len(echantillon)} articles a traiter")

    colonnes_resultat = ["id", "domaine", "vrai_label", "verdict_llm", "correct"]
    resultats = []

    with open(chemin_sortie, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=colonnes_resultat)
        writer.writeheader()

        for i, article in enumerate(echantillon):
            titre = article.get("title", "")
            texte = article.get("text", "")
            source = article.get("domaine", "source inconnue")
            vrai_label = article.get("label", "")

            preuves = chercher_preuves_2(titre, texte)
            time.sleep(1)

            verdict = demander_verdict(titre, texte, source, preuves)
            correct = "oui" if verdict == vrai_label else "non"

            ligne_resultat = {
                "id": article.get("id", ""),
                "domaine": source,
                "vrai_label": vrai_label,
                "verdict_llm": verdict,
                "correct": correct,
            }
            writer.writerow(ligne_resultat)
            resultats.append(ligne_resultat)

            print(f"  [{i + 1}/{len(echantillon)}] {source} -> vrai={vrai_label}, LLM={verdict}, correct={correct}")

    return resultats


**Expérience 1 :**

on va interroger le LLM 3 fois une fois en mentionnant la vraie source originale du Média, la 2e fois en changeant la source, et la 3e fois sans mentionner la source.

In [ ]:
def verdict_n_fois(titre, texte, source, preuves, n=5):
    # Fonction qui appelle le LLM n fois avec les memes parametres
    # et renvoie le verdict 
    # on lance n fois pour chercher une certaine stabilité dans les reponses du modele
    # apres je prends la verdict majoritaire retourné

    verdicts = []
    for i in range(n):
        v = demander_verdict(titre, texte, source, preuves)
        verdicts.append(v)
        
    # compte combien a chaque fois cahque valeur apparait dans la liste
    compte = Counter(verdicts)
    # retourne l'element le plus frequent, [0][0] pour chercher l'etiquette real ou fake
    majorite = compte.most_common(1)[0][0]
    return majorite, verdicts

In [ ]:
def tester_contrefactuel(article, source_2 ,pas_source="source non identifiée"):
    
    # exemple de source peu connue : yournewswire.com
    # exemple de source connue : dailymail.co.uk
    # Je teste un article 3 fois : 
    # une fois avec sa vraie source originale,
    # une fois avec une source oposée au statut de la source originale
    # si la source originale est connue, on changera pour une source peu_connue et virce versa
    # une fois en anonymisant la source en mettant a la place "source non identifiée"
    # mais en gardant -- meme contenu, meme preuves.
    
    titre = article.get("title", "")
    texte = article.get("text", "")
    vraie_source = article.get("domaine", "source inconnue")
    vrai_label = article.get("label", "")

    # On cherche les preuves UNE SEULE FOIS, pour les reutiliser dans les 3 versions
    preuves = chercher_preuves(titre, texte)

    # Version 1 : avec la vraie source 
    verdict_original, n_original = verdict_n_fois(titre, texte, vraie_source, preuves)

    # Version 2 : source changée pour l'opposé de la source originale
    verdict_oppose, n_oppose = verdict_n_fois(titre, texte, source_2, preuves)

    # Version 3 : source rendue anonyme
    verdict_anonyme, n_anonyme = verdict_n_fois(titre, texte, pas_source, preuves)
    

    # flip = oui sil y a au moins un verdict different entre les 3 rendus
    if verdict_original == verdict_oppose == verdict_anonyme:
        flip = "non"
    else:
        flip = "oui"

    return {
        "id": article.get("id", ""),
        "vraie_source": vraie_source,
        "vrai_label": vrai_label,
        "verdict_original": verdict_original,
        "verdict_oppose": verdict_oppose,
        "verdict_anonyme": verdict_anonyme,
        "flip": flip,
    }

In [ ]:
def lancer_test_contrefactuel(chemin_csv, taille_echantillon, chemin_sortie, source_2):
    lignes = lire_csv(chemin_csv)
    echantillon = echantillonner(lignes, taille_echantillon)

    colonnes = ["id", "vraie_source", "vrai_label", "verdict_original", "verdict_oppose" ,"verdict_anonyme", "flip"]
    resultats = []

    with open(chemin_sortie, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=colonnes)
        writer.writeheader()

        for i, article in enumerate(echantillon):
            r = tester_contrefactuel(article,source_2)
            writer.writerow(r)
            resultats.append(r)
            print(f"[{i+1}/{len(echantillon)}] {r['vraie_source']} -> vraiLabel={r['vrai_label']}, original={r['verdict_original']}, oppose={r['verdict_oppose']} ,anonyme={r['verdict_anonyme']}, flip={r['flip']}")
            
    return resultats

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/data-in/"
chemin_out = "/kaggle/working/"

resultats_contrefactuel = lancer_test_contrefactuel(
    chemin_csv=chemin_in + "categorie_B_egalise.csv",
    taille_echantillon=150,
    chemin_sortie=chemin_out + "resultats_contrefactuel_B.csv",
    source_2="yournewswire.com"
)


In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/data-in/"
chemin_out = "/kaggle/working/"

resultats_contrefactuel = lancer_test_contrefactuel(
    chemin_csv=chemin_in + "categorie_D_egalise.csv",
    taille_echantillon=150,
    chemin_sortie=chemin_out + "resultats_contrefactuel_D.csv",
    source_2="dailymail.co.uk"
)


**TEST**

**TEST****TEST**

**TEST**

**TEST**

In [ ]:
article_test = echantillonner(lire_csv(chemin_drive + "categorie_B_egalise.csv"), 1)[0]

titre = article_test.get("title", "")
texte = article_test.get("text", "")
vraie_source = article_test.get("domaine", "source inconnue")

preuves = chercher_preuves_2(titre, texte)

prompt_connu = construire_prompt(titre, texte, vraie_source, preuves)
prompt_anonyme = construire_prompt(titre, texte, "une source non identifiee", preuves)

print("=== PROMPT AVEC SOURCE CONNUE ===")
print(prompt_connu[:300])
print("\n=== PROMPT AVEC SOURCE ANONYME ===")
print(prompt_anonyme[:300])

**TEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEESSSST**

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/data-in/"
chemin_out = "/kaggle/working/"

resultats_test_B = test_groupe(
    chemin_csv=chemin_in + "categorie_B_egalise.csv",
    nom_groupe="Catégorie B - Média connus et peu crédibles - TEST",
    chemin_sortie= chemin_out + "resultats_B_test_v2.csv",
    taille_groupe=150
)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/data-in/"
chemin_out = "/kaggle/working/"

resultats_test_D = test_groupe(
    chemin_csv=chemin_in + "categorie_D_egalise.csv",
    nom_groupe="Catégorie D - Média peu connus et peu crédibles - TEST",
    chemin_sortie= chemin_out + "resultats_D_test_v2.csv",
    taille_groupe=150
)

Résultat de l'experience 1 : pas de biais de notoriété sur l'accuracy globale. Mais il y a un vrai biais de méfiance différentielle : le LLM se montre plus sceptique envers les sources peu connues (il repère mieux leurs fake, mais rejette aussi plus de leurs vrais articles)